# AgentIR on Google Colab

Runs the AgentIR pipeline (agent ↔ simulated user, 10 rounds, Recall@K / Hits@K / BRI) on the
50k VisDial + COCO search space, with Qwen through DashScope for both the questioner and the
simulated user. Same API key as `PlugIR_Colab.ipynb`.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (or better). The GPU is only used
for the BLIP retriever; the LLMs run on DashScope.

What this notebook does:
1. Installs AgentIR from this repo
2. Keeps cache and runs in Colab (downloadable at the end) or, optionally, in Google Drive
3. Downloads the COCO 2017 unlabeled images and checks all 50k search-space images are present
4. Builds the retrieval index (or reuses `cache/corpus_blip.pth` from `PlugIR_Colab.ipynb`)
5. Sanity-checks the index against PlugIR's published BRI, with no LLM calls
6. Runs PlugIR, a random-ranker control, and our answer-divergence ranker on the 50-dialog dev subset
7. Downloads the results

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = 'agentir'   # switch to 'main' once the branch is merged

%cd /content
!git clone -b {BRANCH} https://github.com/nopharshalom/ChatIR-Cornell2026 2>/dev/null || (cd ChatIR-Cornell2026 && git fetch && git checkout {BRANCH} && git pull)
%cd /content/ChatIR-Cornell2026
!pip install -q -e ".[dev]"

In [ ]:
# Offline end-to-end check (synthetic world, mock LLM, ~10 s). If this fails, stop here.
!pytest -q
!agentir smoke --n 20 --output-dir /tmp/smoke | tail -1

## 2. Where to keep cache and runs

`cache/` holds the retrieval index (~200 MB); `runs/` holds one folder per experiment.

- **`USE_DRIVE = False`** (default): everything is **deleted when the session ends**. Run the last cell to download it.
- **`USE_DRIVE = True`**: `cache/` and `runs/` are linked to Google Drive and survive.

The LLM response cache stays on Colab's local disk (SQLite is unreliable on Drive) and is included in the download.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/chatir-cornell2026'
    !mkdir -p "{SAVE_DIR}/cache" "{SAVE_DIR}/runs"
    !rm -rf cache runs && ln -s "{SAVE_DIR}/cache" cache && ln -s "{SAVE_DIR}/runs" runs
else:
    SAVE_DIR = None
    !mkdir -p cache runs
!ls -la cache runs

## 3. Images

About 19 GB, 10–20 minutes, redone in each new Colab session. Same location as `PlugIR_Colab.ipynb` (`/content/data`), so both notebooks share it.

In [ ]:
import os
if not os.path.isdir('/content/data/unlabeled2017'):
    !mkdir -p /content/data
    !wget -q --show-progress -O /content/data/unlabeled2017.zip http://images.cocodataset.org/zips/unlabeled2017.zip
    !unzip -q /content/data/unlabeled2017.zip -d /content/data && rm /content/data/unlabeled2017.zip

# Checks all 50,000 protocol images exist; fills any VisDial targets missing from COCO.
!agentir prepare-data --data-dir /content/data

## 4. Retrieval index

Settings shared by every command below. `configs/base.yaml` pins the protocol and model settings for the whole team.

In [ ]:
B = ('-c configs/base.yaml -c configs/backends/dashscope.yaml '
     '--set data.corpus_dir=/content/data --set retriever.device=cuda '
     '--set agent.llm.cache_path=/content/llm_cache.sqlite '
     '--set user_sim.llm.cache_path=/content/llm_cache.sqlite')

In [ ]:
# Reuse PlugIR_Colab.ipynb's image embeddings if present (upload cache/corpus_blip.pth), else embed from scratch.
if os.path.exists('cache/index_blip.pt'):
    print('index already built')
elif os.path.exists('cache/corpus_blip.pth'):
    !agentir build-index {B} --from-plugir-cache cache/corpus_blip.pth
else:
    !agentir build-index {B}

## 5. Sanity check (no LLM)

Scores PlugIR's released dialogs with our index. **Expect BRI ≈ 0.767** (PlugIR paper, Table 1). If it is far off, the images or the index are wrong; fix that before running anything else.

In [ ]:
!agentir eval-dialogs {B} dialogues/ours_final_q_n_5_recall_hitting_10_thres_low_500_recon_true_referring_true_filtering_true_select_true_reconed.json --split --name check-plugir-released
!agentir eval-dialogs {B} dialogues/VisDial_v1.0_queries_val.json --name check-human-dialogs

## 6. LLM

Add your key in Colab's **Secrets** panel (key icon on the left) as `DASHSCOPE_API_KEY` and turn on notebook access. The default endpoint is the international (Singapore) one; for another region, add `--set agent.llm.base_url=... --set user_sim.llm.base_url=...` to `B`.

In [ ]:
from google.colab import userdata
os.environ['DASHSCOPE_API_KEY'] = userdata.get('DASHSCOPE_API_KEY')

# latency, no reasoning leaking into answers, and image input for the simulated user
!agentir check-llm {B} --image /content/data/unlabeled2017/000000185565.jpg

## 7. Dev-subset runs (first 50 dialogs × 10 rounds)

Each run makes a few thousand API calls; check DashScope pricing first. Runs resume if interrupted: just rerun the cell.
Drop `-c configs/dev.yaml` for the full 2,064 dialogs.

In [ ]:
for exp in ['plugir', 'random_ranker', 'divergence']:
    !agentir run {B} -c configs/experiments/{exp}.yaml -c configs/dev.yaml

In [ ]:
# Side-by-side summary
import glob, json
rows = []
for d in sorted(glob.glob('runs/*/metrics.json')):
    m = json.load(open(d))
    if 'hits' not in m or 'name' not in m:
        continue
    r = m['num_rounds']
    rows.append((m['name'], m['num_sessions'], m['bri'], m['hits']['10'][r], m['recall']['10'][r], m['recall']['1'][r]))
print(f"{'run':<16}{'n':>5}{'BRI':>8}{'Hits@10':>9}{'R@10':>8}{'R@1':>7}   (final round; BRI lower is better)")
for name, n, bri, h10, r10, r1 in rows:
    print(f"{name:<16}{n:>5}{bri:>8.3f}{h10:>9.1f}{r10:>8.1f}{r1:>7.1f}")

## 8. Download results

Zips `runs/`, the LLM cache, and the retrieval index (so next session can skip step 4 by uploading `cache/index_blip.pt`).

In [ ]:
!zip -qr /content/agentir_results.zip runs cache/index_blip.pt /content/llm_cache.sqlite 2>/dev/null; ls -lh /content/agentir_results.zip
from google.colab import files
files.download('/content/agentir_results.zip')